### **Cell 1: Clone Repository and Install Dependencies**
We clone the official `fast-detect-gpt` repository. It requires basic ML libraries like PyTorch and Transformers.

In [ ]:
!git clone https://github.com/baoguangsheng/fast-detect-gpt.git
%cd fast-detect-gpt
!pip install -q transformers torch numpy scipy pandas scikit-learn

### **Cell 2: Initialize Fast-DetectGPT**
Here we load the detector. By default, it uses `gpt-neo-2.7B` for both the sampling and scoring models, which fits comfortably on Kaggle's T4 GPUs while providing excellent speed and accuracy.

In [ ]:
import sys
import torch
import warnings
warnings.filterwarnings('ignore')

# Append the scripts folder to Python path so we can import their code
sys.path.append("scripts")
from local_infer import FastDetectGPT
from types import SimpleNamespace

# Setup model configuration
args = SimpleNamespace(
    sampling_model_name="gpt-neo-2.7B",
    scoring_model_name="gpt-neo-2.7B",
    device="cuda" if torch.cuda.is_available() else "cpu",
    cache_dir="../cache"
)

print("Loading Fast-DetectGPT (gpt-neo-2.7B). This may take a few minutes...")

# Initialize detector
detector = FastDetectGPT(args)

print("Model loaded successfully!")

### **Cell 3: Test on Sample Sentences**

In [ ]:
# Sample inputs
test_sentences = [
    "Yesterday, I went to the local farmer's market and bought some fresh strawberries. The sun was warm, and the vendors were friendly.",
    "Yesterday, I went out with my friends. We had a hangout after 6 or 7 months, and I was very excited for this meetup. we went to several different restaurants but eventually settled for teriaki, which is a japanese restaurant. We order seafood rice, dori fish and chicken dish. I really enjoyed the dori fish with the seas food rice. It was amazing."
]

for text in test_sentences:
    # compute_prob returns: probability of being AI (0 to 1), criterion metric, and number of tokens used
    prob, crit, ntokens = detector.compute_prob(text)
    
    print(f"Text: \"{text}\"")
    print(f"Probability of being AI: {prob * 100:.2f}%")
    print(f"(Raw Criterion Score: {crit:.4f}, Tokens: {ntokens})")
    print("-" * 50)

### **Cell 4: Run Evaluation on AI_Human.csv (Optional)**
Similar to the previous notebooks, this code will run the detector over a balanced sample of your Kaggle dataset.

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm

# 1. Load the dataset
df = pd.read_csv("/kaggle/input/datasets/shanegerami/ai-vs-human-text/AI_Human.csv")

# 2. Sample 500 Human (0.0) and 500 AI-Generated (1.0)
df_human = df[df['generated'] == 0.0].sample(n=500, random_state=42)
df_ai = df[df['generated'] == 1.0].sample(n=500, random_state=42)
df_eval = pd.concat([df_human, df_ai]).sample(frac=1, random_state=42).reset_index(drop=True)

# Fast-DetectGPT is fast but we still truncate to ~300 words to ensure rapid processing
df_eval['text_short'] = df_eval['text'].apply(lambda x: " ".join(str(x).split()[:300]))

predictions = []

print("Running Fast-DetectGPT predictions on 1000 samples...")

for i, row in tqdm(df_eval.iterrows(), total=len(df_eval)):
    text = row['text_short']
    
    try:
        prob, crit, ntokens = detector.compute_prob(text)
        # The paper uses > 50% probability as the threshold for "machine generated"
        is_ai = 1.0 if prob > 0.5 else 0.0
    except Exception as e:
        # Fallback if text is too short or fails
        is_ai = 0.0
        
    predictions.append(is_ai)

df_eval['predicted'] = predictions

# 4. Calculate and Print Metrics
y_true = df_eval['generated']
y_pred = df_eval['predicted']

print("\n" + "="*50)
print("FAST-DETECTGPT PERFORMANCE METRICS (1000 Samples)")
print("="*50)
print(f"Overall Accuracy: {accuracy_score(y_true, y_pred):.4f}\n")
print(classification_report(y_true, y_pred, target_names=["Human (0.0)", "AI-Generated (1.0)"]))


### **Cell 5: Install FastAPI + ngrok**
These packages turn the notebook into a live REST server accessible from the internet.

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest_asyncio

### **Cell 6: Start the API Server + ngrok Tunnel**
This cell launches a FastAPI server on port 8000 and exposes it via ngrok. Copy the printed public URL and paste it into the Verilens popup under **AI Text Detection** → *Test connection*.

> **Keep this cell running.** The server stays alive as long as the Kaggle session is active.

In [ ]:
import nest_asyncio
import uvicorn
import threading
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI(title="Fast-DetectGPT API")

# Allow the Chrome extension (and any browser) to call us across origins.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class TextRequest(BaseModel):
    text: str

@app.get("/health")
def health():
    return {"status": "ok", "model": "fast-detect-gpt / gpt-neo-2.7B"}

@app.post("/detect-text")
async def detect_text(req: TextRequest):
    if not req.text or not req.text.strip():
        return JSONResponse(status_code=400, content={"error": "empty_text"})
    try:
        prob, crit, ntokens = detector.compute_prob(req.text)
        return {
            "ai_probability": round(float(prob), 4),
            "criterion": round(float(crit), 4),
            "tokens": int(ntokens)
        }
    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})

# ---- Start ngrok tunnel, then start uvicorn in a background thread ----
# Set your ngrok auth token to avoid the 2-hour session limit.
# Copy it from lib/config.example.js → NGROK.AUTH_TOKEN and paste below.
NGROK_AUTH_TOKEN = ""  # <-- paste your token here
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

PORT = 8000
tunnel = ngrok.connect(PORT, bind_tls=True)
public_url = tunnel.public_url
print("=" * 60)
print(f"Fast-DetectGPT API is live at: {public_url}")
print(f"  Health: {public_url}/health")
print(f"  Detect: POST {public_url}/detect-text")
print("Paste the base URL into the Verilens popup → AI Text Detection.")
print("=" * 60)

thread = threading.Thread(
    target=uvicorn.run,
    kwargs={"app": app, "host": "0.0.0.0", "port": PORT},
    daemon=True
)
thread.start()
